In [ ]:
import numpy as np
import random
from numba import njit

### Here we calculate the outcomes for varying values of transition probabilities

In [ ]:
# NOTATION: Q is the MUTANT strategy, P is the resident strategy

# We expect vectors of size 5 in here, last element is the probability of initial cooperation

# A1, A2,... each represents a row of the transition matrix

# We should have a 5x5 matrix here for the 5 states CC,CD,DC,DD,A 

# P = [pcc,pcd,pdc,pdd,p0] where p0 is the probability of initial cooperation
def payoff_calculation(P,Q,z2,z4):
    P_trans_CC = 1
    P_trans_CD = z2
    P_trans_DD = z4
    A1 = np.array([P[0]*Q[0]*(P_trans_CC),P[0]*(1-Q[0])*(P_trans_CC),(1-P[0])*Q[0]*(P_trans_CC),(1-P[0])*(1-Q[0])*P_trans_CC,1-P_trans_CC])
    A2 = np.array([P[1]*Q[2]*P_trans_CD,P[1]*(1-Q[2])*P_trans_CD,(1-P[1])*Q[2]*P_trans_CD,(1-P[1])*(1-Q[2])*P_trans_CD,1-P_trans_CD])
    A3 = np.array([P_trans_CD*P[2]*Q[1],P_trans_CD*P[2]*(1-Q[1]),P_trans_CD*(1-P[2])*Q[1],P_trans_CD*(1-P[2])*(1-Q[1]),1-P_trans_CD])
    A4 = np.array([P_trans_DD*P[3]*Q[3],P_trans_DD*P[3]*(1-Q[3]),P_trans_DD*(1-P[3])*Q[3],P_trans_DD*(1-P[3])*(1-Q[3]),1-P_trans_DD])
    A5 = np.array([0,0,0,0,1])
    matrix = np.array([A1,A2,A3,A4,A5])
    v0 = [P[4]*Q[4],P[4]*(1-Q[4]),(1-P[4])*Q[4],(1-P[4])*(1-Q[4]),0]
    V= (1-delta)*np.dot(v0,np.linalg.inv((np.eye(5)-delta*matrix)))  #Stationary state vector/ mean distribution                                                #Normalizing
    pA = np.dot(np.array([R_,S_,T_,P_,Q_]),V)                        #Payoff of first player
    pB = np.dot(np.array([R_,T_,S_,P_,Q_]),V)
    return list([pA,pB,V])                                           #returns the mean distribution and the payoffs of the players

In [ ]:
delta = 0.7      # Continuation probability / payoff discounting parameter
b = 1.7           # benefit term in donation game
c = 1             # Cost term, hence b/c = b

R_ = b-c         # Reward R
S_ = -c          # Sucker's payoff S
T_ = b           # Temptation T
P_ = 0           # mutual defection punishment P
Q_ = 0.1# Absorbing state payoff

init_strat = [0.01,0.01,0.01,0.01,0.01]                          # initial strategy the population will be full off 

In [ ]:
# Calculates the net payoff in a population of N with j mutants
# Notations P: resident, Q: mutant
# Notations PP: payoff of P against P, PQ: payoff of P against Q, and so on


#Returns the payoff of the resident strategy and the mutant Q
def net_payoff(PP,PQ,QP,QQ,N,j):
    Pnet = ((N-j-1)*PP + j*PQ)/(N-1)
    Qnet = ((j-1)*QQ + (N-j)*QP)/(N-1)
    return [Pnet,Qnet]

In [ ]:
# beta is the selection strength
def fixation_prob(P,Q,N,beta,z2,z4):
    P_case = payoff_calculation(P,P,z2,z4)
    P_vec = P_case[2]                       # mean distributuion when P plays against P
    PP = P_case[0]                          #Payoff of P against itself
    mixed= payoff_calculation(P,Q,z2,z4)
    PQ = mixed[0]                           #Payoff of P against Q
    QP = mixed[1]                           #Payoff of Q against P
    Q_case = payoff_calculation(Q,Q,z2,z4)
    Q_vec = Q_case[2]                       # mean distribution when Q vs Q
    QQ = Q_case[0] 
    
    #Calculating the fixation prob using the formula
    S = 1
    factor = 1
    for i in range(1,N):
        pi = net_payoff(PP,PQ,QP,QQ,N,i)
        alpha = np.exp(-beta*(pi[1]-pi[0]))
        factor*=alpha
        S+=factor
    fixprob = 1/S
    stats = [PP,QQ,P_vec,Q_vec]
    return fixprob,stats

In [ ]:
# Calculates time spent in state-1 from mean distribution
def t_state1(vector):
    return 1-vector[4]

# Calculates cooperation rate from mean distribution
def c_rate(vector):
    return (vector[0]+(vector[1]+vector[2])/2)/(1-vector[4])


#Checks if a strategy is partner 
def partner_check(arr,z2,z4,ep=0.1):
    #z2 and z4 are the transition probabilities
    p1 = arr[0]
    p2 = arr[1]
    p3 = arr[2]
    p4 = arr[3]
    p0 = arr[4]
    cond0 = p0-(1-ep)                # p_{0} should be in neighbourhood of 1 or p0>(1-ep) 
    cond1 = p1 -(1-ep)               # p_{1} should be in neighbourhood of 1 or p1>(1-ep) 
    flag = 0
    #Next conditions are from Akin's lemma
    cond2 = (T_ - R_)*(1 - delta)*(1 - (1 - p3)*delta*z2) + (R_ - Q_)*(1 - z2)*(delta*z2*(p2 - p3) - 1) - (R_ - S_)*(1 - delta)*(1 - p2)*delta*z2 
    cond3 = T_ - R_ + delta * (Q_ - T_ + (-1 + delta) * P_ * (-1 + p2) * z2 - Q_ * z2 + delta * Q_ * z2 - delta * p2 * Q_ * z2 + p2 * R_ * z2 + (-1 + p4) * (-R_ + delta * (Q_ - T_) + T_) * z4 + delta * (p2 - p4) * (Q_ - R_) * z2 * z4)
    if cond0>0 and cond1>0 and cond2<0 and cond3<0:
        flag=1        
    return flag
    

# Main function
# Takes mutants in every iteration and checks if it can replace the resident strategy



#parameters: popN is population size, time is number of generations, Beta is selection strength
def selection(z2,z4, popN = 100,time = 500000,Beta = 1):
    #Initial paramters for the run 
    strat = []                                       # Always consists of two elements, [resident,mutant] for each iteration 
    strat.append(init_strat)                        # initial resident population of random population
    z22 = z2
    z44 = z4
    init_state = payoff_calculation(init_strat,init_strat,z22,z44)      # calculating the payoffs of the two strategies 
    Avg_payoff = init_state[0]   
    strat.append(init_strat)                        # initial resident population of random population
    mutant = np.random.uniform(0, 1, 5)             # Drawing five values from 0 to 1 for initial mutant
    
    coop_rate = c_rate(init_state[2])
    T_state1 = t_state1(init_state[2])
    partners = 0
    strat.append(mutant)                           
    #Above code initiates a population of ALL-D with a random mutant to check against for the first time
    temp_C = coop_rate
    temp_s1 = T_state1
    temp_part = partners
    temp_payoff = Avg_payoff
    
    for i in range(time):
        prob,pi = fixation_prob(P = strat[0],Q = strat[1],N=popN,beta = Beta,z2 = z22, z4 = z44)  #Should give the fixation probability of strat Q in resident P population

        
        #Scenario when mutant gets fixed
        if random.random()<prob: 
            del strat[0]                          #Eliminate resident strategy
            temp_payoff = pi[1]
            temp_C = c_rate(pi[3])
            temp_s1 = t_state1(pi[3])
            temp_part = partner_check(strat[0],z22,z44)
        else:
        #Mutant doesn't get selected 
            del strat[1]                          #Eliminate previous mutant
        Avg_payoff+=temp_payoff
        coop_rate+=temp_C
        T_state1+=temp_s1
        partners+=temp_part
        mutant = np.random.uniform(0, 1, 5)   #New mutant introduced
        strat.append(mutant)
    return np.array([coop_rate/(time+1),Avg_payoff/(time+1),T_state1/(time+1), partners/(time+1)])

In [ ]:
# Setting the values of z2 and z4 for testing 



import numpy as np
z_2 = [0.01,0.25,0.5,0.75,0.99,1]
z_4 = z_2 

In [ ]:
""" Testing cooperation rates and other outcomes for varying z_2 and z_4"""

pop = 100
time = 500000
strength = 1

def averager(i,j,trials = 10):
    Results = np.zeros(4)
    for k in range(trials):
        Results+= selection(i,j)
    return Results/trials
    

heatmap  = [[averager(i,j) for j in z_4] for i in z_2]

In [ ]:
heatmap = np.array(heatmap,dtype=object)
np.save("matrix_z2z4_1.npy", heatmap)

In [ ]:
"""For example the cooperation rates for varying z2 and z4 is obtained by accessing the 0th index of the saved heatmap, other outcomes such as partner fraction can be
obtained similarly"""



L = int(len(z_2))
print(L)
coop_rate = np.zeros((L,L))
for i in range(L):
    for j in range(L):
        coop_rate[i,j] = heatmap[i,j][0]